# Fill `results_preprint.tex` numbers from pipeline outputs

> **Not in the submitted manuscript** — convenience autofill for LaTeX drafting. Main submission stats are in `data/shared_data_manuscript_2026/results_valid129/`.

Reads outputs under `analysis/manuscript-2026/` (or `data/shared_data_manuscript_2026/`) and writes `results_preprint_numbers_autofill.txt` in the manuscript folder.

In [1]:
import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd

_cwd = Path.cwd().resolve()
if _cwd.name == "not_in_manuscript":
    _nim = _cwd
elif (_cwd / "analysis/manuscript-2026/not_in_manuscript").exists():
    _nim = _cwd / "analysis/manuscript-2026/not_in_manuscript"
elif _cwd.name == "manuscript-2026" and (_cwd / "not_in_manuscript").exists():
    _nim = _cwd / "not_in_manuscript"
else:
    raise FileNotFoundError(
        "Run from repo root, analysis/manuscript-2026/, or not_in_manuscript/"
    )
if str(_nim) not in sys.path:
    sys.path.insert(0, str(_nim))
from _paths import MANUSCRIPT_DIR

PREPRINT_DIR = MANUSCRIPT_DIR
TEX_PATH = PREPRINT_DIR / "results_preprint.tex"

RESULTS_CANDIDATES = sorted(
    PREPRINT_DIR.glob("main_results_valid129*/results"),
    key=lambda p: p.parent.name,
    reverse=True,
)

seen = set()
RESULTS_CANDIDATES = [p for p in RESULTS_CANDIDATES if not (p in seen or seen.add(p))]
RESULTS_DIR = next((p for p in RESULTS_CANDIDATES if p.exists()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError("Could not find a valid129 results directory under analysis/manuscript-2026")

OUT_PATH = PREPRINT_DIR / "results_preprint_numbers_autofill.txt"
OUT_TABLE_PATH = PREPRINT_DIR / "results_preprint_numbers_table.csv"
print(f"Using RESULTS_DIR: {RESULTS_DIR}")


Using RESULTS_DIR: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/main_results_valid129s_04302026/results


In [2]:
def pick_existing(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these files exist: {[str(p) for p in candidates]}")

paths = {
    "long_tail_by_semantic": pick_existing(
        RESULTS_DIR / "long_tailed_powerlaw_fits_filtered-0.27_valid129.csv",
        RESULTS_DIR / "long_tailed_power_law_fit_by_semantic.csv",
    ),
    "category_cos_clip": RESULTS_DIR / "category_wise_cosine_similarity_clip_filtered-0.27_valid129.csv",
    "category_cos_clip_dino": RESULTS_DIR / "category_wise_cosine_similarity_clip_dinov3_filtered-0.27_valid129.csv",
    "rdm_bv_things": pick_existing(
        RESULTS_DIR / "bv_things_rdm_comparison_summary_filtered-0.27_valid129.csv",
        RESULTS_DIR / "bv_things_rdm_comparison_summary_v2_lowertri_filtered-0.27_valid129.csv",
    ),
    "cluster_strength": RESULTS_DIR / "bv_vs_things_cluster_strength_valid129.csv",
    "cluster_strength_rankcorr": RESULTS_DIR / "bv_vs_things_cluster_strength_rankcorr_valid129.csv",
    "binary_template_corr": RESULTS_DIR / "binary_template_vs_real_rdm_correlations_valid129.csv",
    "pairwise_subject_summary": RESULTS_DIR / "individual_rdm_pairwise_correlation_summary_clip_dinov3_filtered-0.27_valid129.csv",
}

missing = [k for k, p in paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected result files: {missing}")

for k, p in paths.items():
    print(f"{k}: {p.name}")

df_long_sem = pd.read_csv(paths["long_tail_by_semantic"])
if "cdi_semantic" not in df_long_sem.columns and "semantic_category" in df_long_sem.columns:
    # Harmonize newer long-tail output schema with the downstream code.
    df_long_sem = df_long_sem.rename(columns={"semantic_category": "cdi_semantic"})
if "cdi_semantic" in df_long_sem.columns:
    # Newer files label semantic rows as e.g. semantic_animals.
    df_long_sem["cdi_semantic"] = (
        df_long_sem["cdi_semantic"].astype(str).str.replace("^semantic_", "", regex=True)
    )

df_cos_clip = pd.read_csv(paths["category_cos_clip"])
df_cos_both = pd.read_csv(paths["category_cos_clip_dino"])
df_rdm = pd.read_csv(paths["rdm_bv_things"])
df_cluster = pd.read_csv(paths["cluster_strength"])
df_cluster_rank = pd.read_csv(paths["cluster_strength_rankcorr"])
df_binary = pd.read_csv(paths["binary_template_corr"])
df_pair_summary = pd.read_csv(paths["pairwise_subject_summary"])


long_tail_by_semantic: long_tailed_powerlaw_fits_filtered-0.27_valid129.csv
category_cos_clip: category_wise_cosine_similarity_clip_filtered-0.27_valid129.csv
category_cos_clip_dino: category_wise_cosine_similarity_clip_dinov3_filtered-0.27_valid129.csv
rdm_bv_things: bv_things_rdm_comparison_summary_filtered-0.27_valid129.csv
cluster_strength: bv_vs_things_cluster_strength_valid129.csv
cluster_strength_rankcorr: bv_vs_things_cluster_strength_rankcorr_valid129.csv
binary_template_corr: binary_template_vs_real_rdm_correlations_valid129.csv
pairwise_subject_summary: individual_rdm_pairwise_correlation_summary_clip_dinov3_filtered-0.27_valid129.csv


In [3]:
values = {}

semantic_order = [
    "clothing", "furniture_rooms", "household", "toys", "body_parts",
    "food_drink", "outside", "vehicles", "animals"
]
alpha_by_sem = df_long_sem.set_index("cdi_semantic")["alpha"].reindex(semantic_order)
values["long_tail_alpha_by_semantic"] = {k: float(v) for k, v in alpha_by_sem.dropna().items()}
values["long_tail_alpha_semantic_mean"] = float(alpha_by_sem.dropna().mean())

values["cos_clip_mean"] = float(df_cos_clip["cosine_similarity"].mean())
values["cos_clip_min"] = float(df_cos_clip["cosine_similarity"].min())
values["cos_clip_max"] = float(df_cos_clip["cosine_similarity"].max())
values["cos_clip_n"] = int(df_cos_clip.shape[0])

clip_col = "cosine_similarity_clip" if "cosine_similarity_clip" in df_cos_both.columns else "clip_cosine_similarity"
dino_col = "cosine_similarity_dinov3" if "cosine_similarity_dinov3" in df_cos_both.columns else "dinov3_cosine_similarity"
clip_vals = df_cos_both[clip_col].to_numpy()
dino_vals = df_cos_both[dino_col].to_numpy()
values["clip_vs_dino_categorywise_pearson_r"] = float(np.corrcoef(clip_vals, dino_vals)[0, 1])

rdm_lookup = df_rdm.set_index("model")
values["rdm_bv_vs_things_spearman_clip"] = float(rdm_lookup.loc["clip", "spearman_r"])
values["rdm_bv_vs_things_spearman_dinov3"] = float(rdm_lookup.loc["dinov3", "spearman_r"])
if "babydinov3" in rdm_lookup.index:
    values["rdm_bv_vs_things_spearman_babydinov3"] = float(rdm_lookup.loc["babydinov3", "spearman_r"])

# Positive dissimilarity direction to match manuscript reporting.
bin_sel = df_binary[["model", "source", "spearman_rho_template_dissim_vs_rdm"]].copy()
for model in ["clip", "dinov3", "babydinov3"]:
    for source in ["babyview", "things"]:
        val = float(bin_sel[(bin_sel.model == model) & (bin_sel.source == source)]["spearman_rho_template_dissim_vs_rdm"].iloc[0])
        values[f"binary_template_rho_{source}_{model}"] = val

for _, row in df_cluster.iterrows():
    model = str(row["model"]).lower()
    values[f"cluster_delta_diff_{model}"] = float(row["delta_diff_bv_minus_things"])
    values[f"cluster_delta_ci_low_{model}"] = float(row["boot_ci_low"])
    values[f"cluster_delta_ci_high_{model}"] = float(row["boot_ci_high"])

for _, row in df_cluster_rank.iterrows():
    model = str(row["model"]).lower()
    values[f"cluster_rank_spearman_{model}"] = float(row["spearman_rho"])
    values[f"cluster_rank_spearman_p_{model}"] = float(row["spearman_p"])
    values[f"cluster_rank_n_clusters_{model}"] = int(row["n_clusters_used"])

pair_idx = df_pair_summary.set_index("group")
values["top8_avg_pearson_clip"] = float(pair_idx.loc["top8_densest_subjects_clip", "avg_pearson_r"])
values["top8_sd_pearson_clip"] = float(pair_idx.loc["top8_densest_subjects_clip", "std_pearson_r"])
values["top8_avg_pearson_dinov3"] = float(pair_idx.loc["top8_densest_subjects_dinov3", "avg_pearson_r"])
values["top8_sd_pearson_dinov3"] = float(pair_idx.loc["top8_densest_subjects_dinov3", "std_pearson_r"])
values["top8_avg_spearman_clip"] = float(pair_idx.loc["top8_densest_subjects_clip", "avg_spearman_rho"])
values["top8_sd_spearman_clip"] = float(pair_idx.loc["top8_densest_subjects_clip", "std_spearman_rho"])
values["top8_avg_spearman_dinov3"] = float(pair_idx.loc["top8_densest_subjects_dinov3", "avg_spearman_rho"])
values["top8_sd_spearman_dinov3"] = float(pair_idx.loc["top8_densest_subjects_dinov3", "std_spearman_rho"])

values


{'long_tail_alpha_by_semantic': {'clothing': 1.2298183199449189,
  'furniture_rooms': 1.9767694894668228,
  'household': 1.931819812664791,
  'toys': 1.6786357766958968,
  'body_parts': 2.364041708700343,
  'food_drink': 1.676359057611733,
  'outside': 1.9837652159484305,
  'vehicles': 3.1253367504033958,
  'animals': 1.6900412245445848},
 'long_tail_alpha_semantic_mean': 1.9618430395534352,
 'cos_clip_mean': 0.6089347891242057,
 'cos_clip_min': 0.1412660563057558,
 'cos_clip_max': 0.8013006456906926,
 'cos_clip_n': 129,
 'clip_vs_dino_categorywise_pearson_r': 0.6950562536435041,
 'rdm_bv_vs_things_spearman_clip': 0.5536067100159698,
 'rdm_bv_vs_things_spearman_dinov3': 0.4017801989711168,
 'binary_template_rho_babyview_clip': 0.3533065789076423,
 'binary_template_rho_things_clip': 0.4077710641057646,
 'binary_template_rho_babyview_dinov3': 0.3435483942183597,
 'binary_template_rho_things_dinov3': 0.2744854803022957,
 'cluster_delta_diff_clip': 0.0561935334689773,
 'cluster_delta_ci_lo

In [4]:
tex = TEX_PATH.read_text()
xx_count = len(re.findall(r"\bXX\b", tex))

OUT_PARAGRAPHS_PATH = PREPRINT_DIR / "results_preprint_paragraphs_autofill.txt"

lines = [
    "Autofill values for results_preprint.tex",
    "====================================",
    f"Using results directory: {RESULTS_DIR}",
    f"XX placeholders found: {xx_count}",
    "",
    "Long-tailed distributions:",
    f"- alpha (semantic mean proxy across listed CDI groups): {values['long_tail_alpha_semantic_mean']:.2f}",
    "- alpha by CDI semantic group:",
]
for k in semantic_order:
    if k in values["long_tail_alpha_by_semantic"]:
        lines.append(f"  - {k}: {values['long_tail_alpha_by_semantic'][k]:.2f}")

lines += [
    "",
    "Category-wise BabyView vs THINGS similarity (CLIP):",
    f"- n categories: {values['cos_clip_n']}",
    f"- cosine min/max/mean: {values['cos_clip_min']:.2f} / {values['cos_clip_max']:.2f} / {values['cos_clip_mean']:.2f}",
    f"- cross-model category-wise Pearson r (CLIP vs DINOv3): {values['clip_vs_dino_categorywise_pearson_r']:.2f}",
    "",
    "Between-category RDM correlations (BabyView vs THINGS):",
    f"- CLIP Spearman rho: {values['rdm_bv_vs_things_spearman_clip']:.2f}",
    f"- DINOv3 Spearman rho: {values['rdm_bv_vs_things_spearman_dinov3']:.2f}",
    "",
    "Binary CDI template alignment (Spearman, dissimilarity coding):",
    f"- CLIP rho_BV={values['binary_template_rho_babyview_clip']:.3f}, rho_THINGS={values['binary_template_rho_things_clip']:.3f}",
    f"- DINOv3 rho_BV={values['binary_template_rho_babyview_dinov3']:.3f}, rho_THINGS={values['binary_template_rho_things_dinov3']:.3f}",
    f"- BabyDINOv3 rho_BV={values['binary_template_rho_babyview_babydinov3']:.3f}, rho_THINGS={values['binary_template_rho_things_babydinov3']:.3f}",
    "",
    "CDI cluster-strength difference (BabyView minus THINGS):",
    f"- CLIP delta={values['cluster_delta_diff_clip']:.3f}, 95% CI [{values['cluster_delta_ci_low_clip']:.3f}, {values['cluster_delta_ci_high_clip']:.3f}]",
    f"- DINOv3 delta={values['cluster_delta_diff_dinov3']:.3f}, 95% CI [{values['cluster_delta_ci_low_dinov3']:.3f}, {values['cluster_delta_ci_high_dinov3']:.3f}]",
    "",
    "Across-cluster BV-vs-THINGS agreement:",
    f"- CLIP Spearman rho={values['cluster_rank_spearman_clip']:.3f}, p={values['cluster_rank_spearman_p_clip']:.3g}, n_clusters={values['cluster_rank_n_clusters_clip']}",
    f"- DINOv3 Spearman rho={values['cluster_rank_spearman_dinov3']:.3f}, p={values['cluster_rank_spearman_p_dinov3']:.3g}, n_clusters={values['cluster_rank_n_clusters_dinov3']}",
    "",
    "Top-8 family pairwise RDM consistency (Spearman):",
    f"- CLIP mean={values['top8_avg_spearman_clip']:.3f}, SD={values['top8_sd_spearman_clip']:.3f}",
    f"- DINOv3 mean={values['top8_avg_spearman_dinov3']:.3f}, SD={values['top8_sd_spearman_dinov3']:.3f}",
    "",
    "Manual follow-up still needed for placeholders without direct manuscript-2026 result tables:",
    "- dataset-level detection count/precision text at the beginning of Results",
    "- VQA comparison sentence (r=.72, overlapping n=99)",
    "- top-8 family hours/age range fields",
]

paragraphs = [
    (
        "To quantify distributional shape, we fit power-law functions to category frequencies. "
        f"Across the 129 categories, the estimated exponent was $\\alpha={values['long_tail_alpha_semantic_mean']:.2f}$. "
        "Long-tailed structure was also present within CDI semantic groups, with "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['clothing']:.2f}$ (clothing), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['furniture_rooms']:.2f}$ (furniture/rooms), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['household']:.2f}$ (household), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['toys']:.2f}$ (toys), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['body_parts']:.2f}$ (body parts), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['food_drink']:.2f}$ (food/drink), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['outside']:.2f}$ (outside), "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['vehicles']:.2f}$ (vehicles), and "
        f"$\\alpha={values['long_tail_alpha_by_semantic']['animals']:.2f}$ (animals)."
    ),
    (
        "Category-level BabyView-versus-THINGS similarity varied widely in CLIP space, from "
        f"{values['cos_clip_min']:.2f} to {values['cos_clip_max']:.2f} "
        f"(mean $\\cos(\\theta)={values['cos_clip_mean']:.2f}$; $n={values['cos_clip_n']}$ categories). "
        "Category rankings were moderately aligned between embedding spaces, with "
        f"CLIP-versus-DINOv3 category-wise correlation of $r={values['clip_vs_dino_categorywise_pearson_r']:.2f}$"
        "."
    ),
    (
        "Between-category representational geometry was correlated between BabyView and THINGS in both feature spaces "
        f"(CLIP Spearman $\\rho={values['rdm_bv_vs_things_spearman_clip']:.2f}$; "
        f"DINOv3 Spearman $\\rho={values['rdm_bv_vs_things_spearman_dinov3']:.2f}$). "
        "Superordinate CDI clustering was stronger in BabyView than in THINGS: "
        f"CLIP $\\Delta={values['cluster_delta_diff_clip']:.3f}$, 95\\% CI "
        f"[{values['cluster_delta_ci_low_clip']:.3f}, {values['cluster_delta_ci_high_clip']:.3f}]; "
        f"DINOv3 $\\Delta={values['cluster_delta_diff_dinov3']:.3f}$, 95\\% CI "
        f"[{values['cluster_delta_ci_low_dinov3']:.3f}, {values['cluster_delta_ci_high_dinov3']:.3f}]."
    ),
    (
        "Template-based checks supported this pattern: binary CDI template alignment was "
        f"CLIP ($\\rho_{{BV}}={values['binary_template_rho_babyview_clip']:.3f}$, "
        f"$\\rho_{{THINGS}}={values['binary_template_rho_things_clip']:.3f}$) and "
        f"DINOv3 ($\\rho_{{BV}}={values['binary_template_rho_babyview_dinov3']:.3f}$, "
        f"$\\rho_{{THINGS}}={values['binary_template_rho_things_dinov3']:.3f}$). "
        "BV-vs-THINGS cluster-strength ranks were also correlated across CDI groups "
        f"(CLIP $\\rho={values['cluster_rank_spearman_clip']:.3f}$, "
        f"$p={values['cluster_rank_spearman_p_clip']:.3g}$; "
        f"DINOv3 $\\rho={values['cluster_rank_spearman_dinov3']:.3f}$, "
        f"$p={values['cluster_rank_spearman_p_dinov3']:.3g}$)."
    ),
    (
        "Across the top eight densest families, pairwise RDM similarity remained high in both spaces "
        f"(CLIP mean Spearman $r={values['top8_avg_spearman_clip']:.3f}$, SD={values['top8_sd_spearman_clip']:.3f}; "
        f"DINOv3 mean Spearman $r={values['top8_avg_spearman_dinov3']:.3f}$, SD={values['top8_sd_spearman_dinov3']:.3f}), "
        "indicating consistent category structure across families despite idiosyncratic environments."
    ),
]

paragraph_text = "\n\n".join(paragraphs) + "\n"

# Save text summary used for manuscript drafting.
OUT_PATH.write_text("\n".join(lines) + "\n")
OUT_PARAGRAPHS_PATH.write_text(paragraph_text)

# Save a tidy table of all computed numeric outputs for downstream checks.
flat_rows = []
for key, val in values.items():
    if isinstance(val, dict):
        for subkey, subval in val.items():
            flat_rows.append({"metric": f"{key}.{subkey}", "value": float(subval)})
    else:
        flat_rows.append({"metric": key, "value": val})

df_results_table = pd.DataFrame(flat_rows).sort_values("metric").reset_index(drop=True)
df_results_table.to_csv(OUT_TABLE_PATH, index=False)

print("Wrote:", OUT_PATH)
print("Wrote:", OUT_TABLE_PATH)
print("Wrote:", OUT_PARAGRAPHS_PATH)
print("\n".join(lines[:20]))
print("\nParagraph preview:\n")
print("\n\n".join(paragraphs[:2]))
print(df_results_table.head(10))


Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/results_preprint_numbers_autofill.txt
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/results_preprint_numbers_table.csv
Autofill values for results_preprint.tex
Using results directory: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/main_results_valid129s_04302026/results
XX placeholders found: 12

Long-tailed distributions:
- alpha (semantic mean proxy across listed CDI groups): 1.96
- alpha by CDI semantic group:
  - clothing: 1.23
  - furniture_rooms: 1.98
  - household: 1.93
  - toys: 1.68
  - body_parts: 2.36
  - food_drink: 1.68
  - outside: 1.98
  - vehicles: 3.13
  - animals: 1.69

Category-wise BabyView vs THINGS similarity (CLIP):
- n categories: 129
                                metric     value
0    binary_template_rho_babyview_clip  0.353307
1  binary_template_rho_babyview_dinov3  0.343548
2      binary_templat